In [ ]:
import kagglehub

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd

data = pd.read_csv(f"{path}/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
data.head()

In [ ]:
# Task 3: Write your code here:
data.info()

In [ ]:
# Task 4: Write your code here:
data.describe()

In [ ]:
# Task 1: Write your code here:
print(data.isnull().sum())
data.fillna(data.median(), inplace=True)  # filled null data with median

In [ ]:
# Task 2: Write your code here:
duplicates = data.duplicated().sum()
print(f'Duplicates: {duplicates}')
data.drop_duplicates(inplace=True)

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder
# Identify categorical columns
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

data

In [ ]:
# Task 4: Write your code here:
scaler = StandardScaler()
numerical_cols = data.select_dtypes(include=['int64', 'float64']).columns.drop("Target").tolist()
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

In [ ]:
# Task 5: Write your code here:
target_counts = data['Target'].value_counts()
print(target_counts)
print('Is the target imbalanced?', target_counts.min() / target_counts.max() < 0.1)


In [ ]:
%pip install catboost

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

X = data.drop('Target', axis=1)
y = data['Target']


In [ ]:
# Task 2,3,4,5: Write your code here:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

# Task 3 and 4: Train CatBoostClassifier and evaluate
for train_index, test_index in kfold.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, silent=True)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Task 4: Evaluate using F1 Score
    f1_scores.append(f1_score(y_test, y_pred))

# Task 5: Print the averaged F1 score across all folds
print('Average F1 Score:', sum(f1_scores) / len(f1_scores))


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

# Task 1: Plot feature importance from your trained model
feature_importances = model.get_feature_importance()
features = X.columns

# Create a DataFrame for visualization
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Plot
plt.figure(figsize=(12, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.show()

In [ ]:
# Task 2: Write your code here:
golden_feature = importance_df.iloc[0]['Feature']
print('Golden Feature:', golden_feature)

In [ ]:
#Task Bonus: Write your code here:
X_golden = X[[golden_feature]]

#Run the same KFold loop with this single feature
f1_scores_golden = []

for train_index, test_index in kfold.split(X_golden, y):
    X_train, X_test = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model_golden = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, silent=True)
    model_golden.fit(X_train, y_train)

    y_pred = model_golden.predict(X_test)
    f1_scores_golden.append(f1_score(y_test, y_pred))

#Print and compare the accuracy with the full model
print('Average F1 Score with Golden Feature:', sum(f1_scores_golden) / len(f1_scores_golden))
